---
title: Basic XML Parsing with ElementTree and EAD
---

:::{important} Learning Outcomes in this Section
- A better understanding of EAD as a metadata scheme
- Importing and using Python's XML library, ElementTree, often known as ET
- Loading and Processing XML with ET
- Practice and expand skills to examine metadata encoded in XML
:::


Python provides useful tools to work with XML.
This section of the book introduces useful libraries and modules that are useful in this regard, as well as some of the techniques and approaches that you can reuse.
Along the way, examples from various standard cultural heritage metadata schemes provide case studies, including archival metadata in Encoded Archival Description (EAD) and resource metadata in Metadata Object Description Schema (MODS).
You will find both in various digital collections.

This section assumes readers are familiar with the basic syntax and requirements of XML.
As reviewed earlier (see [](/part01/04-open-data-formats.md#xml-a-quick-introduction)), XML is a standard markup language derived from SGML. If you are looking for more detail, the Text Encoding Initiative (TEI) offers a "[Gentle Introduction to XML](https://tei-c.org/release/doc/tei-p5-doc/en/html/SG.html)" that covers the primary features of XML as a data structure and for encoding text in XML.

## Encoded Archival Description

Encoded Archival Description (better known as EAD) is a content and structure standard that provides rules for making archival finding aids computer processable.
Developed in the late 1990s, the standard is now in its fourth version and maintained by the Society of American Archivists (SAA) and Library of Congress. The standard documentation for EAD is maintained by the Library of Congress at <https://loc.gov/ead/>. 
The information that goes into EAD is defined by the SAA's [_Describing Archives: A Content Standard_](http://www2.archivists.org/standards/DACS). This section will address EAD version 3, which is further described at this toolkit <https://github.com/saa-ead-roundtable/ead3-toolkit>.

### A Sample EAD Record

Before using Python's XML tools,
take a look at [](#ead-ex01) (below).
This brief example of simplified EAD
illustrates some of the tags and data that are used later on.

```{code} xml
:label: ead-ex01
:filename: /data/ead-superior-sample.xml (simplified)
:linenos:
:emphasize-lines: 2, 3, , 6-8, 11, 12, 21, 22
:caption: This excerpt shows a few brief snippets from a sample EAD document created for the fictional "Superior Papers" Collection. The full file can be found in the `data` directory. Note that some content was removed for readability, so the excerpt is not directly equivalent to the source.
<?xml version="1.0" encoding="UTF-8"?>
<ead xmlns="http://ead3.archivists.org/schema/" audience="external">
    <control countryencoding="iso3166-1" dateencoding="iso8601" langencoding="iso639-2b">
        <recordid instanceurl="http://jajohnst.si.umich.edu/fake-ead.xml">1234</recordid>
        <filedesc>
            <titlestmt>
                <titleproper>A Finding Aid for the Superior Papers</titleproper>
            </titlestmt>
. . .
        </filedesc>
    </control>
    <archdesc level="collection" audience="external">
. . .
        <bioghist>This collection is a superior collection of papers for a superior organization and individual.</bioghist>
. . .
        <dsc dsctype="otherdsctype" audience="external">
            <c01 level="series">Business Records</c01>
            <c02 level="series">Correspondence</c02>
            <c03 level="file" audience="internal">Personal File</c03>
        </dsc>
    </archdesc>
</ead>
```


While a full explanation of EAD is beyond the scope of this book (useful overviews are given in the [SAA's EAD3 Starter Kit](https://github.com/saa-ead-roundtable/ead3-toolkit), among other places),
note some features of the sample data ([](#ead-ex01) above).
The first line contains an XML declaration, which tells the parser this is an XML document; it also contains attributes that indicate the version of the XML and the character encoding.
On the next line (line 2), the root `ead` tag opens; it closes on last line of the example (line 22). Note that the root tag also contains an `xmlns` attribute, which names the EAD schema and links this document to the structure and definitions of that namespace; this means, in other words, that all of this XML document follows the namespace of EAD, unless otherwise indicated. The [`ead` tag is defined in the EAD tag library](https://www.loc.gov/ead/EAD3taglib/EAD3-TL-eng.html#elem-ead), which states this is the "required root element of an EAD instance."

Farther along in the example ([](#ead-ex01)), note the two subelements: `control` (lines 3&ndash;11) and `archdesc` (lines 12&ndash;22). As the EAD documentation clarifies, these are both required, they are also the only allowed elements within the main `ead` tag, and they must occur in this order. The [`control` element](https://www.loc.gov/ead/EAD3taglib/EAD3-TL-eng.html#elem-control) is for "for recording bibliographic and administrative information about an EAD instance." In other words, this largely contains information about the EAD document: who created, when, and how it is managed, as well as descriptive information about what it describes. The [`archdesc` element](https://www.loc.gov/ead/EAD3taglib/EAD3-TL-eng.html#elem-archdesc) "binds together all of the archival descriptive information in an EAD instance." In other words, this is where you would look for the information about the materials or collection described by the EAD, including things like context (the who, what, where, and when of the materials or collection described), the organization extent of a collection (listing its main series or parts), and information describing those elements, if described. Note also the [`titleStmt` element](https://www.loc.gov/ead/EAD3taglib/EAD3-TL-eng.html#elem-titlestmt), which describes the name and authors of a finding aid (see [](#ead-ex01), lines 6&ndash;8).

:::{tip} Learning about EAD Elements
The above section explains the XML features and a few tags in the sample EAD document,
but it does not explain every tag. Note, for example, the uses of `did`, `unittitle`, `unitdate`, `dsc`, `c01`, and so on. These are other EAD tags and if you work
further with EAD, it is important to learn about them as well.
To do this, it can be useful to consult the definitions of these tags in the EAD documentation. One useful starting place is the ["EAD tag library" (the library for version 3 is linked here)](https://www.loc.gov/ead/EAD3taglib/EAD3-TL-eng.html), which provides an alphbetically sorted list of all the valid tags available in EAD.
The list defines each tag and explains the rules for using each one.

_Nota bene_: Ruth Kitchin Tillman's [EADiva](https://eadiva.com/) resource is an even more accessible tool for looking up EAD tags.
:::

Now that you have a sense of some of the elements to expect in an EAD document,
and their usage and structure, let's get to the data!

## Getting Started with ElementTree

The standard tool for working with XML in Python is the [ElementTree module](https://docs.python.org/3/library/xml.etree.elementtree.html). While this requires a special import statement, it is considered part of Python's standard library.

### Set Up: Loading Data and the ElementTree Module

This section illustrates loading the sample data and then imports the ElementTree module.

### Import Sample EAD Data

First, import the data. The code below will appear in many other sections, too,
and illustrates a consistent way to load sample data that helps to maintain
consistent paths between systems.

:::{attention} Data Import
As throughout the book, all data files can be found in the `data` directory
which accompanies this book and can be found in book's GitHub repository.
This structure, which more or less packages the data alongside the worked examples in this notebook,
allows you to run the notebook yourself if you download the full repository from GitHub,
while using the same commands and producing the same results. 

The following cell illustrates a consistent way to reference the data files
without using complex paths later on.
:::

In [1]:
from pathlib import Path

DATA = Path('..', 'data')
OUT = Path('..', 'output')

EAD_sample = DATA / 'ead-superior-sample.xml'

### Import ElementTree

Next, import the ElementTree module. Although it requires a special import call,
this is the a "built-in" Python module, which means it is part of standard Python distributions.
ElementTree is sometimes known as _eTree_ and most commonly referred to in code as _ET_.
The ElementTree module will accomplish most of the things that you need to do with XML to manage metadata.

The ElementTree module is typically aliased _ET_ and imported as follows. This allows you to call the module by typing `ET` rather than the whole string. 

In [2]:
import xml.etree.ElementTree as ET

## Loading (parsing) the XML

This section demonstrates how to load and parse XML data with eTree, and the functions that will be useful. To accomplish initial XML parsing these are:

### Functions for loading and parsing

* `.parse()` - creates a python object that we can manipulate with ElementTree
* `.getroot()` - structures an ElementTree object according to the root element that you set

Use the following sequence to load the sample EAD (with the `.parse()` function), assign it to a variable (using the `.getroot()` function), and finally to confirm the data is loaded using a basic print statement.

In [9]:
tree = ET.parse(EAD_sample)
root = tree.getroot()

print(root)
print(len(root))

<Element '{http://ead3.archivists.org/schema/}ead' at 0x109f69530>
2


Note that the response is a list of Elements, which are identified with their tag names and location in memory.

To get a better sense of the data, the string methods are useful since these allow the output of an Element item as text. The `ead` element is quite long, so the following limits the data printed by slicing only the first 150 characters.  

In [12]:
print(ET.tostring(root)[:150])
print(len(ET.tostring(root)))

b'<ns0:ead xmlns:ns0="http://ead3.archivists.org/schema/" audience="external">\n    <ns0:control countryencoding="iso3166-1" dateencoding="iso8601" lange'
1692


As the second line of the output shows, the parser processed 1,692 characters in the full document.

The above steps to parse could be made more efficient by parsing the XML directly to a variable, for example as `ead = ET.parse(EAD_sample).getroot()`.
These two steps can, however, be useful to separate. The reason is that these two actions return different kinds of data. The `.parse()` function returns an `ElementTree` data type, which is a data structure
that reflects an entire XML structure hierarchy that can be manipulated as a whole and is particularly useful to write out (i.e., save) the data in certain ways.
The `.getroot()` element, on the other hand, returns and `Element` data type, which reflects a particular section of XML. In other words, it is like a node, which knows the name of the node, its contents and attributes, and what subelements it contains.
Most of the work to manage metadata involves working with the various element-level pieces of the whole tree.
So most quality control and data entry work, which involves specific pieces of information in the tree
can be accomplished by working with elements.
Operations that involve the entire tree of an XML document, like validation and saving the data,
may require full ElementTree data. 

:::{hint} What is Parsing?
When you import a data file, like the MODS records encoded in XML, _parse_ is a fancy
way to refer to opening the file in a way that allows the computer to understand
the file in a particular way.
In this case, the BeautifulSoup library is "parsing" the received data as marked up
text. Because you know this is XML that follows the MODS standard,
you can then ask Python to process the data in certain ways.
If you didn't tell Python how to parse the file, it would just see it as
raw data without being able to name the various elements and data inside it.

We want to be able to refer to specific parts of the data by their element names,
which requires this parsing step.
:::

## Navigating and Modifying the Tree

This section first illustrates basic actions that support navigation through and identification of particulr data or types of data within the tree. Then, it demonstrates how to add or modify data. Subsequent sections demonstrate more advanced querying with XPath, writing files, and validation. 

### Accessing Basic Properties of Elements

Once the data is parsed, it is possible to extract the values of elements, any associated attributes, and the contents nested within the element markers. Basic methods, which can be appended to any `Element` object, are as follows:

- `.tag` returns the name of the tag, aka the element name;
- `.attrib` identifies an Element's attributes, and makes the available as a dictionary; and
- `.text` returns the nested content, whether text or other elements, that is wrapped between the opening and closing tags.

A `for` loop provides an initial tool to observe these properties.

In [13]:
for element in root:
    print(element)

<Element '{http://ead3.archivists.org/schema/}control' at 0x109f698a0>
<Element '{http://ead3.archivists.org/schema/}archdesc' at 0x109f692b0>


As you can see in the above responses, the loop identifies two elements.
This is working with the sample data, in which we observed root `ead` tag in the sample data contains two subelements: `control` and `archdesc`. The pointy brackets (`< >`) around each line indicate this is an Element data object, and the string of characters with curly braces communicate a reference to the full tag name.
As shorthand, the basic tag name is easier to use; the first element above, then, can be thought of as the `control` element. 

:::{hint} Why are the element names so long?
When Python parsed this XML tree, it learned that it is an EAD document.
This is indicated in [](#ead-ex01) on the `ead` tag's `xmlns` attribute, which indicate an XML _namespace_.
We will look more at namespaces later on.
When a namespace is specified and recognized by the parser, however, it includes a reference to namespace
by prepending the helpful (though rather long) reference to the namespace in curly braces.
Above, for example `{http://ead3.archivists.org/schema/}control` indicates the `control` element.
While for general purposes, it is sufficient to refer to the element by its tag name, the full element name
helps to make its source and usage unambiguous. This full name, which is constructed by appending the namespace and element name, is also called a [qualified name or QName](wiki:QName).

Although unwieldy, QNames are an important part of XML parsing and are critical in tracking an element's operative scheme or vocabulary.
:::

ElementTree treats all of the elements as data objects. If you are interested in the content,
it is more useful to see this as readable text. For this, use the `.tostring()` and `.fromstring()` functions.
These functions convert the binary data elements into plain text.

In [15]:
print(ET.tostring(root)[:500])

b'<ns0:ead xmlns:ns0="http://ead3.archivists.org/schema/" audience="external">\n    <ns0:control countryencoding="iso3166-1" dateencoding="iso8601" langencoding="iso639-2b">\n        <ns0:recordid instanceurl="http://jajohnst.si.umich.edu/fake-ead.xml">1234</ns0:recordid>\n        <ns0:filedesc>\n            <ns0:titlestmt>\n                <ns0:titleproper>A Finding Aid for the Superior Papers</ns0:titleproper>\n            </ns0:titlestmt>\n            <ns0:publicationstmt>\n                <ns0:publish'


While not exactly the same, the above response is similar to line 2 in [](#ead-ex01). Because the string does not have an attached namespace, the tag name `ns0:ead` indicates a "namespace 0" for the unidentified schema.

#### The `.tag` method

The `.tag` method displays the tag name, and the attributes associated with the element are stored in a dictionary that can be called with the `.attrib` method. Above, the `for` loop showed the sub elements of the `root` variable.
If you want to know the tag name of that element, however, use `root.tag`:

In [16]:
root.tag

'{http://ead3.archivists.org/schema/}ead'

#### The `.attrib` method

The `.attrib` method calls an element's attributes as a dictionary. If there are no elements, this is an empty dictionary. Below, the `ead` root's attributes show `@audience`.

In [17]:
root.attrib

{'audience': 'external'}

To get a closer view of the elements in tree, a `for` loop allows the traversal of the root element's contents. Below, the loop also returns the tag names and attributes.

In [19]:
for element in root:
    print(element.tag,' -- ',element.attrib)

{http://ead3.archivists.org/schema/}control  --  {'countryencoding': 'iso3166-1', 'dateencoding': 'iso8601', 'langencoding': 'iso639-2b'}
{http://ead3.archivists.org/schema/}archdesc  --  {'level': 'collection', 'audience': 'external'}


The `.text` method is useful if you are working with an element that wraps text.
Many elements, however, only contain other elements, which is stored by the parser as data.
For example, the root element returns only a newline character and some spaces (see below).
The `.text` will be used more later when we work with elements that contain text.

In [28]:
root.text

'\n    '

### Useful functions to find and retrieve data

The ElementTree module supports a few basic functions that support finding data in the tree, both elements, attributes, and wrapped text; functions that allow for looping through parts of the tree to locate data according to specific criteria; and known item location or data retrieval. We will look at each of these in turn:

- `.find()` - returns the first match to an element name, provided as a string or variable; 
- related, if you want to locate multiple elements or search through the tree, `.findall()` will return a list of all matching elements;
- `.iter()` - creates an iterable, which can be used in a loop. Useful to find all of the elements within a given tree or element structure, not just the ones at the current level or matching a specific element name;
- `.get()` - allows you to get specified attributes

#### The `.iter()` function

When looking into the `.attrib` method (see [](#the-attrib-method) above), a simple `for` loop listed the contents of the `root` variable. If you are interested in looking through the full tree structure, rather than an invidual element, these functions are more useful. For example, to see all of the elements in the tree, the `iter()` function returns a variable that you can loop through.

In [36]:
for element in root.iter():
    print(element.tag,' -- ',element.attrib)

{http://ead3.archivists.org/schema/}ead  --  {'audience': 'external'}
{http://ead3.archivists.org/schema/}control  --  {'countryencoding': 'iso3166-1', 'dateencoding': 'iso8601', 'langencoding': 'iso639-2b'}
{http://ead3.archivists.org/schema/}recordid  --  {'instanceurl': 'http://jajohnst.si.umich.edu/fake-ead.xml'}
{http://ead3.archivists.org/schema/}filedesc  --  {}
{http://ead3.archivists.org/schema/}titlestmt  --  {}
{http://ead3.archivists.org/schema/}titleproper  --  {}
{http://ead3.archivists.org/schema/}publicationstmt  --  {}
{http://ead3.archivists.org/schema/}publisher  --  {}
{http://ead3.archivists.org/schema/}date  --  {'normal': '2022-09-01'}
{http://ead3.archivists.org/schema/}archdesc  --  {'level': 'collection', 'audience': 'external'}
{http://ead3.archivists.org/schema/}did  --  {}
{http://ead3.archivists.org/schema/}repository  --  {}
{http://ead3.archivists.org/schema/}corpname  --  {}
{http://ead3.archivists.org/schema/}part  --  {}
{http://ead3.archivists.org/sc

#### Using `.find()`

The `.iter()` function is useful if you need to loop through any provided element in its entirety.
In many cases, however, you may be looking for a particular tag, attribute, or another known quantity.
To select known elements like this, use the `.find()` function, which returns the first object that matches your request. This makes sense if looking at the `control` element since there is only one in any EAD document. Recall, above, that `control` is a required EAD element, which must be the first child element of the root `ead`, and it may only occur once.

When working with subelements, it is often useful to isolate them from the full tree. The block below illustrates this technique by assigning the result of `.find()` to a variable named `control`.
The resulting variable can be used as an element.
Note that the following also uses the full elementtree assigned above to `tree` rather than the root element. 

In [32]:
control = tree.find('{http://ead3.archivists.org/schema/}control')
print(control.attrib)

{'countryencoding': 'iso3166-1', 'dateencoding': 'iso8601', 'langencoding': 'iso639-2b'}


At this point, a `for` loop is the best way to examine Element objects.
The technique is limited, however, in that it only returns direct child elements.
The only option to get those descendants is to move through the tree step by step.
We will investigate more efficient ways of looking through the tree later on.

In [47]:
for element in control:
    print(element.tag,' -- ',element.attrib)

{http://ead3.archivists.org/schema/}recordid  --  {'instanceurl': 'http://jajohnst.si.umich.edu/fake-ead.xml'}
{http://ead3.archivists.org/schema/}filedesc  --  {}


Note above that the loop only displays the direct descendants of `control`, namely `recordid` and `filedesc`. It does not display any elements lower down in the hierarchy like `titlestmt` or `titleproper`. (See [](#ead-ex01).)

#### A first use of `.findall()`

Many elements will occur more than once in a given tree. When looking multiple elements, the `.findall()` function, which operates similarly, is a better choice.
For example, the above list of all tags produced by `.iter()` shows multiple `part` elements. XML is a very literal format, and at present, the data structure only knows that the `root` variable is an `ead` element and contains the `control` and `archdesc` children. To search within those elements, use the syntax `.//`, which tells ElementTree to look at the root (indicated with `.` just like the current directory in a command line path) and all of its descendant objects (`//`). This is a special example of a query language called XPath, which is covered in more detail later. For now, a test query could look like:

In [ ]:
parts = tree.findall('.//{http://ead3.archivists.org/schema/}part')
print(parts)


[<Element '{http://ead3.archivists.org/schema/}part' at 0x109f6b2e0>, <Element '{http://ead3.archivists.org/schema/}part' at 0x109f6a660>]


/var/folders/33/zdynstwn0m9f31kg0y6bj49w0000gp/T/ipykernel_52485/916976575.py:1: FutureWarning: This search is broken in 1.3 and earlier, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/{http://ead3.archivists.org/schema/}part'
  parts = tree.findall('//{http://ead3.archivists.org/schema/}part')


:::{hint} First use of XPath
The argument here is a modified XPath selector, which is how we will guide the function to the elements that we want to see in the tree. XPath is the subject of the next section, see [](/part02/xml-02-xpath.ipynb).
:::

#### Using `.get()`

Once we find the elements, use the `.get()` method to take a look at the attributes associated with a particular element. 

Continuing with the `control` element, get the value of a particular attribute. For example, `countryencoding`.

In [46]:
countryCode = control.get('countryencoding')

print(f'Country encoding is according to: {countryCode}')

Country encoding is according to: iso3166-1


## Summary

This section introduced the basics of parsing XML with the ElementTree module in Python, also known as _etree_ or _ET_.
The section illustrated a basic pattern for loading an XML file.
Next, the section demonstrated how to identify the basic components of an element, including its name (`.tag`),
attributes (`.attrib`), and its contents (`.text`).
Finally,

The following sections introduce more powerful ways to search through XML, modifying and write XML documents, and how to validate against external standards. This section provides a full toolbox that supports working with, analyzing, and modifying XML.

:::{tip}
**Activities/Lab**

1. Find an EAD example that interests you. Many are available from University of Michigan, the Library of Congress, or other archival collections. Use the skills outlined above to parse, explore, and process the EAD as data. List some elements, make a modification, and write out the file. Accompany the file with a written explanation of your process and a reflection on what you learned. If you used a Jupyter notebook, that should be part of your product. If you used a generative AI tool, identify the tool and model used, and attach a full output (in plain text or JSON) that shows your prompts, the responses, and how you interacted with the tool.
:::